In [1]:
import pandas as pd
print("ok")

ok


In [2]:
import pandas as pd

df = pd.read_csv("students.csv")
df.head()

,student_id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,1,17,male,diploma,2.78,92.9,yes,7.4,poor,coaching,low,hard,58.9
1,2,23,other,bca,3.37,64.8,yes,4.6,average,online videos,medium,moderate,54.8
2,3,22,male,b.sc,7.88,76.8,yes,8.5,poor,coaching,high,moderate,90.3
3,4,20,other,diploma,0.67,48.4,yes,5.8,average,online videos,low,moderate,29.7
4,5,20,female,diploma,0.89,71.6,yes,9.8,poor,coaching,low,moderate,43.7


In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
import joblib

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("students.csv")

# =========================
# 2. DEFINE COLUMNS
# =========================
cat_cols = [
    "gender", "course", "internet_access",
    "sleep_quality", "study_method",
    "facility_rating", "exam_difficulty"
]

num_cols = [
    "age", "study_hours", "class_attendance", "sleep_hours"
]

target_col = "exam_score"

# =========================
# 3. ENCODE CATEGORICAL DATA (CORRECT WAY)
# =========================
encoder = OrdinalEncoder()

df[cat_cols] = encoder.fit_transform(df[cat_cols])

# =========================
# 4. PREPARE FEATURES
# =========================
X = df[cat_cols + num_cols]
y = df[target_col]

# =========================
# 5. TRAIN MODEL
# =========================
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# =========================
# 6. GET FEATURE IMPORTANCE (SAW WEIGHTS)
# =========================
weights = model.feature_importances_

# Normalize weights (important for SAW)
weights = weights / weights.sum()

# Convert to dictionary (for readability)
weight_dict = dict(zip(X.columns, weights))

# =========================
# 7. SAVE EVERYTHING (NO NEED RETRAIN AFTER REBOOT)
# =========================
joblib.dump(model, "model.pkl")
joblib.dump(weights, "weights.pkl")
joblib.dump(encoder, "encoder.pkl")
df.to_pickle("processed_data.pkl")

# =========================
# 8. PRINT RESULT
# =========================
print("\n=== FEATURE WEIGHTS (SAW) ===")
for k, v in weight_dict.items():
    print(f"{k}: {v:.4f}")


=== FEATURE WEIGHTS (SAW) ===
gender: 0.0131
course: 0.0257
internet_access: 0.0054
sleep_quality: 0.0420
study_method: 0.0451
facility_rating: 0.0300
exam_difficulty: 0.0122
age: 0.0283
study_hours: 0.5755
class_attendance: 0.1549
sleep_hours: 0.0679


In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("students.csv")

# =========================
# 2. DEFINE COLUMNS
# =========================
cat_cols = [
    "gender", "course", "internet_access",
    "sleep_quality", "study_method",
    "facility_rating", "exam_difficulty"
]

num_cols = [
    "age", "study_hours", "class_attendance", "sleep_hours"
]

remove_cols = ["gender"]  # you can add "course" if needed

criteria = [c for c in (cat_cols + num_cols) if c not in remove_cols]

# =========================
# 3. ENCODE CATEGORICAL
# =========================
encoder = OrdinalEncoder()
df[cat_cols] = encoder.fit_transform(df[cat_cols])

# =========================
# 4. WEIGHTS (FROM ML)
# =========================
weights = {
    "gender": 0.0131,
    "course": 0.0257,
    "internet_access": 0.0054,
    "sleep_quality": 0.0420,
    "study_method": 0.0451,
    "facility_rating": 0.0300,
    "exam_difficulty": 0.0122,
    "age": 0.0283,
    "study_hours": 0.5755,
    "class_attendance": 0.1549,
    "sleep_hours": 0.0679
}

# Convert to array aligned with columns
weight_array = np.array([weights[col] for col in criteria])

# Cap dominant weights (prevent study_hours domination)
max_cap = 0.4
weight_array = np.minimum(weight_array, max_cap)

# Normalize again
weight_array = weight_array / weight_array.sum()

# =========================
# 5. DEFINE BENEFIT & COST
# =========================
benefit_cols = [
    "study_hours", "class_attendance", "sleep_hours",
    "sleep_quality", "study_method", "facility_rating",
    "internet_access"
]

cost_cols = [
    "exam_difficulty", "age"
]

# =========================
# 6. NORMALIZATION (SAW)
# =========================
df_norm = df.copy()

for col in criteria:
    if col in benefit_cols:
        df_norm[col] = df[col] / df[col].max()
    elif col in cost_cols:
        df_norm[col] = (df[col].max() - df[col]) / (df[col].max() - df[col].min())
    else:
        # treat as benefit by default
        df_norm[col] = df[col] / df[col].max()

# =========================
# 7. CALCULATE SAW SCORE
# =========================
df["saw_score"] = np.dot(df_norm[criteria], weight_array)

# =========================
# 8. RANKING
# =========================
df_ranked = df.sort_values(by="saw_score", ascending=False)

# =========================
# 9. OUTPUT
# =========================
print("\n=== TOP STUDENTS (SAW RANKING) ===")
print(df_ranked[["student_id", "saw_score"]].head(10))

# Save result
df_ranked.to_csv("saw_ranking.csv", index=False)


=== TOP STUDENTS (SAW RANKING) ===
       student_id  saw_score
6834         6835   0.961367
15214       15215   0.936106
8671         8672   0.935888
11845       11846   0.919527
17733       17734   0.919348
14287       14288   0.919314
1949         1950   0.918032
8782         8783   0.915567
18456       18457   0.915382
17757       17758   0.913659


In [6]:
# =========================
# 10. DEFINE PASS / FAIL (GROUND TRUTH)
# =========================
df["actual_result"] = df["exam_score"].apply(lambda x: "PASS" if x >= 60 else "FAIL")

# =========================
# 11. DEFINE SAW PREDICTION
# =========================
# Use median as threshold (balanced approach)
threshold = df["saw_score"].median()

df["saw_result"] = df["saw_score"].apply(lambda x: "PASS" if x >= threshold else "FAIL")

# =========================
# 12. COMPARE RESULTS
# =========================
df["match"] = df["actual_result"] == df["saw_result"]

# Accuracy
accuracy = df["match"].mean() * 100

print(f"\nSAW vs Actual Accuracy: {accuracy:.2f}%")

# =========================
# 13. CONFUSION SUMMARY
# =========================
summary = pd.crosstab(df["actual_result"], df["saw_result"])

print("\n=== CONFUSION MATRIX ===")
print(summary)

# =========================
# 14. SAVE RESULT
# =========================
df.to_csv("saw_validation.csv", index=False)


SAW vs Actual Accuracy: 78.02%

=== CONFUSION MATRIX ===
saw_result     FAIL  PASS
actual_result            
FAIL           7278  1675
PASS           2722  8325


In [7]:
import pandas as pd

df = pd.read_csv("students.csv")

num_cols = [
    "age", "study_hours", "class_attendance", "sleep_hours"
]

min_max = {}

for col in num_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    min_max[col] = {"min": min_val, "max": max_val}

# Print nicely
for col, vals in min_max.items():
    print(f"{col}: min={vals['min']}, max={vals['max']}")

age: min=17, max=24
study_hours: min=0.08, max=7.91
class_attendance: min=40.6, max=99.4
sleep_hours: min=4.1, max=9.9


In [8]:
cat_cols = [
    "gender", "course", "internet_access",
    "sleep_quality", "study_method",
    "facility_rating", "exam_difficulty"
]

categories = {}

for col in cat_cols:
    unique_vals = sorted(df[col].dropna().unique())
    categories[col] = unique_vals
    print(f"\n{col}:")
    print(unique_vals)


gender:
['female', 'male', 'other']

course:
['b.com', 'b.sc', 'b.tech', 'ba', 'bba', 'bca', 'diploma']

internet_access:
['no', 'yes']

sleep_quality:
['average', 'good', 'poor']

study_method:
['coaching', 'group study', 'mixed', 'online videos', 'self-study']

facility_rating:
['high', 'low', 'medium']

exam_difficulty:
['easy', 'hard', 'moderate']


In [11]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv("students.csv")

# =========================
# 1. REMOVE UNUSED COLUMNS
# =========================
df = df.drop(columns=["student_id", "gender", "course"])

# =========================
# 2. MAP CATEGORICAL (SAFE ONES ONLY)
# =========================
df["internet_access"] = df["internet_access"].map({"no": 0, "yes": 1})

df["sleep_quality"] = df["sleep_quality"].map({
    "poor": 0, "average": 1, "good": 2
})

df["facility_rating"] = df["facility_rating"].map({
    "low": 0, "medium": 1, "high": 2
})

df["exam_difficulty"] = df["exam_difficulty"].map({
    "easy": 0, "moderate": 1, "hard": 2
})

# =========================
# 3. ONE-HOT study_method
# =========================
df = pd.get_dummies(df, columns=["study_method"], drop_first=True)

# =========================
# 4. DEFINE X, y
# =========================
X = df.drop(columns=["exam_score"])
y = df["exam_score"]

# =========================
# 5. FORCE NUMERIC (IMPORTANT FIX)
# =========================
X = X.astype(float)

# =========================
# 6. ADD CONSTANT
# =========================
X = sm.add_constant(X)

# =========================
# 7. FIT MODEL
# =========================
model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             exam_score   R-squared:                       0.732
Model:                            OLS   Adj. R-squared:                  0.732
Method:                 Least Squares   F-statistic:                     4554.
Date:                Fri, 24 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:08:10   Log-Likelihood:                -73996.
No. Observations:               20000   AIC:                         1.480e+05
Df Residuals:                   19987   BIC:                         1.481e+05
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const               